# Proposed Model — CatBoost v1

We use CatBoostRegressor to capture nonlinear patterns in sales. Inputs are
15 shared calendar features and three categorical features: country, store,
and product.

We use the shared preprocessing and remove rows with missing num_sold.
The model uses the original target, fixed parameters, and seed 42.
Negative predictions are clipped to zero. No tuning or early stopping is used.

We evaluate with the shared MAPE (%) on expanding-window folds for
2014, 2015, and 2016, plus a 2014–2016 holdout trained on 2010–2013.
Results are compared with seasonal_naive_last_year from results/metrics.csv.

## Reproducibility

Place train.csv and test.csv in data/ and keep the baseline metrics in
results/metrics.csv. Install dependencies in your virtual environment:

```bash
python -m pip install -r requirements.txt
python -m pip install ipykernel nbclient
```

Run all cells in order using this environment. A fresh-kernel run reproduced
all four scores without adding duplicate CSV rows.

Tested: Python 3.14.4, NumPy 2.5.3, pandas 3.0.5, CatBoost 1.2.10.
Versions are not pinned.

## 1. Imports

In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
candidates = [
    current_dir,
    current_dir.parent,
    current_dir / "sticker-sales-forecasting",
]

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "src" / "preprocessing.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(f"Project root not found from: {current_dir}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import catboost

from src.preprocessing import load_data, preprocess_data, drop_missing_target
from src.features import add_date_features
from src.validation import expanding_window_splits, temporal_train_val_split
from src.metrics import mape
from src.model import (
    MODEL_NAME,
    MODEL_VERSION,
    FEATURE_COLS,
    CATEGORICAL_FEATURES,
    DEFAULT_PARAMS,
    TARGET_TRANSFORM,
    fit_model,
    predict_model,
)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("CatBoost:", catboost.__version__)
print("Model:", MODEL_NAME)
print("Version:", MODEL_VERSION)
print("Target transform:", TARGET_TRANSFORM)

Python: 3.11.9
NumPy: 2.4.6
pandas: 3.0.5
CatBoost: 1.2.10
Model: catboost_log
Version: v2
Target transform: log1p


## 2. Data Preparation

In [2]:
train_raw, test_raw = load_data(
    PROJECT_ROOT / "data" / "train.csv",
    PROJECT_ROOT / "data" / "test.csv",
)

train = preprocess_data(train_raw)
missing_target_count = int(train["num_sold"].isna().sum())

train = drop_missing_target(train)
train = add_date_features(train)

assert train[FEATURE_COLS].notna().all().all()
assert train["num_sold"].notna().all()
assert (train["num_sold"] > 0).all()
assert all(
    train[column].map(lambda value: isinstance(value, str)).all()
    for column in CATEGORICAL_FEATURES
)

print("Rows with missing target removed:", missing_target_count)
print("Prepared feature matrix:", train[FEATURE_COLS].shape)

Rows with missing target removed: 8871
Prepared feature matrix: (221259, 18)


## 3. Temporal Validation

In [3]:
folds = list(expanding_window_splits(train))
long_train, long_val = temporal_train_val_split(
    train,
    val_start_date="2014-01-01",
)

split_summary = []

splits_to_check = [
    (f"fold_{index}_{val_part['date'].dt.year.iloc[0]}", train_part, val_part)
    for index, (train_part, val_part) in enumerate(folds, start=1)
]
splits_to_check.append(("long_horizon", long_train, long_val))

for name, train_part, val_part in splits_to_check:
    assert train_part["date"].max() < val_part["date"].min(), name
    assert set(train_part["id"]).isdisjoint(val_part["id"]), name
    assert train_part["num_sold"].notna().all(), name
    assert val_part["num_sold"].notna().all(), name

    split_summary.append({
        "split": name,
        "train_start": train_part["date"].min().date(),
        "train_end": train_part["date"].max().date(),
        "val_start": val_part["date"].min().date(),
        "val_end": val_part["date"].max().date(),
        "train_rows": len(train_part),
        "val_rows": len(val_part),
    })

print(pd.DataFrame(split_summary).to_string(index=False))
print("Temporal split checks passed.")

       split train_start  train_end  val_start    val_end  train_rows  val_rows
 fold_1_2014  2010-01-01 2013-12-31 2014-01-01 2014-12-31      126026     31823
 fold_2_2015  2010-01-01 2014-12-31 2015-01-01 2015-12-31      157849     31643
 fold_3_2016  2010-01-01 2015-12-31 2016-01-01 2016-12-31      189492     31767
long_horizon  2010-01-01 2013-12-31 2014-01-01 2016-12-31      126026     95233
Temporal split checks passed.


## 4. Expanding-Window Evaluation

In [4]:
train_part, val_part = folds[0]

print("Training fold_1_2014")

model_2014 = fit_model(train_part)
predictions_2014 = predict_model(model_2014, val_part)

print("Training completed.")
print("Prediction min:", predictions_2014.min())
print("Prediction max:", predictions_2014.max())
print("Prediction mean:", predictions_2014.mean())
print("Negative predictions:", int((predictions_2014 < 0).sum()))
print("All predictions finite:", bool(np.isfinite(predictions_2014).all()))

mape_2014 = mape(val_part["num_sold"].to_numpy(), predictions_2014)
print(f"MAPE 2014: {mape_2014:.6f}%")

Training fold_1_2014
Training completed.
Prediction min: 4.666542003048453
Prediction max: 5112.789462173784
Prediction mean: 821.4909514419384
Negative predictions: 0
All predictions finite: True
MAPE 2014: 7.334349%


In [5]:
log_predictions_2014 = np.asarray(
    model_2014.predict(folds[0][1][FEATURE_COLS]),
    dtype=float,
)

unclipped_predictions_2014 = np.expm1(log_predictions_2014)

unclipped_mape_2014 = mape(
    folds[0][1]["num_sold"].to_numpy(),
    unclipped_predictions_2014,
)

negative_count = int((unclipped_predictions_2014 < 0).sum())

assert np.allclose(
    predictions_2014,
    np.maximum(unclipped_predictions_2014, 0.0),
)

print("Log prediction min:", log_predictions_2014.min())
print("Log prediction max:", log_predictions_2014.max())
print(
    "Negative predictions after inverse transform:",
    negative_count,
)
print(f"MAPE before clipping: {unclipped_mape_2014:.6f}%")
print(f"MAPE after clipping: {mape_2014:.6f}%")
print("Inverse-transform check passed.")

Log prediction min: 1.734579055684077
Log prediction max: 8.539695986089665
Negative predictions after inverse transform: 0
MAPE before clipping: 7.334349%
MAPE after clipping: 7.334349%
Inverse-transform check passed.


In [6]:
fold_scores = [
    {
        "fold": "fold_1_2014",
        "mape": mape_2014,
    }
]

for fold_number, (train_part, val_part) in enumerate(folds[1:], start=2):
    year = int(val_part["date"].dt.year.iloc[0])
    fold_name = f"fold_{fold_number}_{year}"

    print(f"\nTraining {fold_name}")

    fold_model = fit_model(train_part)
    predictions = predict_model(fold_model, val_part)
    score = mape(val_part["num_sold"].to_numpy(), predictions)

    print("Prediction min:", predictions.min())
    print("Prediction max:", predictions.max())
    print("Prediction mean:", predictions.mean())
    print("Negative predictions:", int((predictions < 0).sum()))
    print("All predictions finite:", bool(np.isfinite(predictions).all()))
    print(f"MAPE: {score:.6f}%")

    fold_scores.append({
        "fold": fold_name,
        "mape": score,
    })

fold_metrics = pd.DataFrame(fold_scores)
mean_expanding_mape = float(fold_metrics["mape"].mean())

print("\nExpanding-window results:")
print(fold_metrics.to_string(index=False))
print(f"Mean expanding-window MAPE: {mean_expanding_mape:.6f}%")


Training fold_2_2015
Prediction min: 4.618806093349014
Prediction max: 4456.703474162181
Prediction mean: 776.3997483536285
Negative predictions: 0
All predictions finite: True
MAPE: 13.599239%

Training fold_3_2016
Prediction min: 4.48961731927351
Prediction max: 3477.295598334333
Prediction mean: 691.2258889537804
Negative predictions: 0
All predictions finite: True
MAPE: 6.565157%

Expanding-window results:
       fold      mape
fold_1_2014  7.334349
fold_2_2015 13.599239
fold_3_2016  6.565157
Mean expanding-window MAPE: 9.166248%


## 5. Long-Horizon Evaluation

In [7]:
print("Training long_horizon")

long_model = fit_model(long_train)
long_predictions = predict_model(long_model, long_val)

long_horizon_mape = mape(
    long_val["num_sold"].to_numpy(),
    long_predictions,
)

print("Prediction min:", long_predictions.min())
print("Prediction max:", long_predictions.max())
print("Prediction mean:", long_predictions.mean())
print("Negative predictions:", int((long_predictions < 0).sum()))
print("All predictions finite:", bool(np.isfinite(long_predictions).all()))
print(f"Long-horizon MAPE: {long_horizon_mape:.6f}%")

Training long_horizon
Prediction min: 4.6072050919721494
Prediction max: 5112.789462173784
Prediction mean: 823.480828178821
Negative predictions: 0
All predictions finite: True
Long-horizon MAPE: 14.943025%


## 6. Baseline Comparison

In [8]:
metrics_path = PROJECT_ROOT / "results" / "metrics.csv"
saved_metrics = pd.read_csv(metrics_path)

baseline_names = [
    "group_median",
    "seasonal_naive_last_year",
    "ridge_calendar_one_hot",
]

reference_model_names = baseline_names + [
    "catboost_raw",
]

reference_metrics = saved_metrics.loc[
    saved_metrics["model"].isin(reference_model_names)
    & saved_metrics["version"].eq("v1")
].copy()

expected_folds = {
    "expanding_window": {
        "fold_1_2014",
        "fold_2_2015",
        "fold_3_2016",
    },
    "long_horizon": {"2014_2016"},
}

for model_name in reference_model_names:
    for scheme, expected in expected_folds.items():
        rows = reference_metrics.loc[
            reference_metrics["model"].eq(model_name)
            & reference_metrics["validation_scheme"].eq(scheme)
        ]

        assert len(rows) == len(expected), (model_name, scheme)
        assert set(rows["fold"]) == expected, (model_name, scheme)

comparison = (
    reference_metrics
    .groupby(["model", "validation_scheme"])["mape"]
    .mean()
    .unstack("validation_scheme")
)

comparison.loc[
    MODEL_NAME,
    "expanding_window"
] = mean_expanding_mape

comparison.loc[
    MODEL_NAME,
    "long_horizon"
] = long_horizon_mape

print(comparison.round(6).to_string())

best_baseline = "seasonal_naive_last_year"

difference_vs_baseline = (
    comparison.loc[MODEL_NAME]
    - comparison.loc[best_baseline]
)

difference_vs_v1 = (
    comparison.loc[MODEL_NAME]
    - comparison.loc["catboost_raw"]
)

print(
    "\nMAPE difference versus seasonal naive "
    "(percentage points):"
)
print(
    difference_vs_baseline.round(6).to_string()
)

print(
    "\nMAPE difference versus CatBoost v1 "
    "(percentage points):"
)
print(
    difference_vs_v1.round(6).to_string()
)

validation_scheme         expanding_window  long_horizon
model                                                   
catboost_raw                     18.244561     22.522954
group_median                     16.319486     17.460368
ridge_calendar_one_hot           17.859827     23.907846
seasonal_naive_last_year         13.273855     17.019122
catboost_log                      9.166248     14.943025

MAPE difference versus seasonal naive (percentage points):
validation_scheme
expanding_window   -4.107607
long_horizon       -2.076097

MAPE difference versus CatBoost v1 (percentage points):
validation_scheme
expanding_window   -9.078312
long_horizon       -7.579929


## 7. Save Results

In [9]:
import json

experiment_params = {
    **model_2014.get_params(),
    "features": FEATURE_COLS,
    "target_transform": TARGET_TRANSFORM,
    "prediction_postprocessing": "exmp1_then_clip_min_0",
    "catboost_version": catboost.__version__,
}

params_json = json.dumps(experiment_params, sort_keys=True)

evaluation_results = [
    (
        "expanding_window",
        row["fold"],
        train_part,
        val_part,
        float(row["mape"]),
    )
    for row, (train_part, val_part) in zip(fold_scores, folds, strict=True)
]

evaluation_results.append(
    (
        "long_horizon",
        "2014_2016",
        long_train,
        long_val,
        float(long_horizon_mape),
    )
)

records = []

for scheme, fold, train_part, val_part, score in evaluation_results:
    records.append({
        "model": MODEL_NAME,
        "version": MODEL_VERSION,
        "validation_scheme": scheme,
        "fold": fold,
        "train_period": (
            f"{train_part['date'].min().date()} to "
            f"{train_part['date'].max().date()}"
        ),
        "validation_period": (
            f"{val_part['date'].min().date()} to "
            f"{val_part['date'].max().date()}"
        ),
        "mape": round(score, 6),
        "params": params_json,
    })

proposed_metrics = pd.DataFrame(records)

assert list(proposed_metrics.columns) == list(saved_metrics.columns)
assert len(proposed_metrics) == 4
assert np.isfinite(proposed_metrics["mape"].to_numpy()).all()
assert not proposed_metrics.duplicated(
    ["model", "version", "validation_scheme", "fold"]
).any()

print(proposed_metrics.drop(columns="params").to_string(index=False))
print("\nParameters:")
print(json.dumps(experiment_params, indent=2, sort_keys=True))
print("\nMetrics prepared. No files written.")

       model version validation_scheme        fold             train_period        validation_period      mape
catboost_log      v2  expanding_window fold_1_2014 2010-01-01 to 2013-12-31 2014-01-01 to 2014-12-31  7.334349
catboost_log      v2  expanding_window fold_2_2015 2010-01-01 to 2014-12-31 2015-01-01 to 2015-12-31 13.599239
catboost_log      v2  expanding_window fold_3_2016 2010-01-01 to 2015-12-31 2016-01-01 to 2016-12-31  6.565157
catboost_log      v2      long_horizon   2014_2016 2010-01-01 to 2013-12-31 2014-01-01 to 2016-12-31 14.943025

Parameters:
{
  "allow_writing_files": false,
  "cat_features": [
    "country",
    "store",
    "product"
  ],
  "catboost_version": "1.2.10",
  "depth": 6,
  "features": [
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "quarter",
    "is_weekend",
    "time_idx",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",


In [10]:
current_metrics = pd.read_csv(metrics_path)
original_bytes = metrics_path.read_bytes()

assert list(current_metrics.columns) == list(proposed_metrics.columns)

own_rows = (
    current_metrics["model"].eq(MODEL_NAME)
    & current_metrics["version"].eq(MODEL_VERSION)
)

if own_rows.any():
    sort_columns = ["validation_scheme", "fold"]

    existing = (
        current_metrics.loc[own_rows]
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )
    expected = (
        proposed_metrics
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        existing,
        expected,
        check_dtype=False,
        check_exact=True,
    )
    print("Identical results already exist. No files changed.")
else:
    with metrics_path.open("a", encoding="utf-8", newline="") as output:
        if original_bytes and not original_bytes.endswith((b"\n", b"\r")):
            output.write("\n")

        proposed_metrics.to_csv(
            output,
            index=False,
            header=False,
            lineterminator="\n",
        )

    assert metrics_path.read_bytes().startswith(original_bytes)
    print("Added 4 proposed-model rows. Existing rows preserved.")

updated_metrics = pd.read_csv(metrics_path)
print("Total metric rows:", len(updated_metrics))

Identical results already exist. No files changed.
Total metric rows: 24


## 8. Conclusions

CatBoost v2 with a log1p-transformed target achieved 9.17% mean
expanding-window MAPE and 14.94% long-horizon MAPE.

Compared with CatBoost v1, the mean expanding-window MAPE improved
from 18.24% to 9.17%, while long-horizon MAPE improved from 22.52%
to 14.94%.

The v2 model also outperformed the strongest baseline,
seasonal_naive_last_year, which achieved 13.27% expanding-window MAPE
and 17.02% long-horizon MAPE.

The only modeling change from v1 was the target transformation:
num_sold was transformed with log1p before training, and predictions
were converted back with expm1 before evaluation. The feature set,
CatBoost hyperparameters, temporal validation splits, and MAPE metric
remained unchanged.

This result shows that the log-target transformation substantially
improves CatBoost performance for this forecasting task. The next
improvement stage can focus on hyperparameter tuning and additional
leakage-safe features.

## 9. CatBoost v3: Hyperparameter Tuning

CatBoost v2 significantly improved both expanding-window and long-horizon
MAPE by using a log1p-transformed target.

For v3, the target transformation, feature set, validation splits, and
evaluation metric remain unchanged. We tune a small set of CatBoost
hyperparameters using only the expanding-window validation folds.

The long-horizon holdout is not used for hyperparameter selection and will
be evaluated only after the best configuration is selected.

In [11]:
tuning_candidates = [
    {
        "name": "v2_reference",
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,
    },
    {
        "name": "shallower",
        "iterations": 500,
        "depth": 4,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,
    },
    {
        "name": "deeper",
        "iterations": 500,
        "depth": 8,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,
    },
    {
        "name": "slow_lr_800",
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 3,
    },
    {
        "name": "slow_lr_1000",
        "iterations": 1000,
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 3,
    },
    {
        "name": "fast_lr",
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.10,
        "l2_leaf_reg": 3,
    },
    {
        "name": "more_regularization",
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 10,
    },
    {
        "name": "medium_regularization",
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5,
    },
    {
        "name": "shallow_slow",
        "iterations": 1000,
        "depth": 4,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5,
    },
    {
        "name": "deep_slow",
        "iterations": 800,
        "depth": 8,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5,
    },
]

pd.DataFrame(tuning_candidates)

,name,iterations,depth,learning_rate,l2_leaf_reg
0,v2_reference,500,6,0.05,3
1,shallower,500,4,0.05,3
2,deeper,500,8,0.05,3
3,slow_lr_800,800,6,0.03,3
4,slow_lr_1000,1000,6,0.03,3
5,fast_lr,500,6,0.10,3
6,more_regularization,500,6,0.05,10
7,medium_regularization,800,6,0.05,5
8,shallow_slow,1000,4,0.03,5
9,deep_slow,800,8,0.03,5


In [12]:
import time

train_part, val_part = folds[0]

test_params = {
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3,
}

start = time.perf_counter()

test_model = fit_model(
    train_part,
    params=test_params,
)

test_predictions = predict_model(
    test_model,
    val_part,
)

elapsed = time.perf_counter() - start

print(f"One reference training took: {elapsed:.1f} seconds")

One reference training took: 49.1 seconds


In [ ]:
tuning_results = []

for candidate in tuning_candidates:
    candidate_name = candidate["name"]

    params = {
        key: value
        for key, value in candidate.items()
        if key != "name"
    }

    print(f"\n{'=' * 60}")
    print(f"Testing: {candidate_name}")
    print(params)

    candidate_scores = []

    for fold_number, (train_part, val_part) in enumerate(folds, start=1):
        year = int(val_part["date"].dt.year.iloc[0])

        model = fit_model(
            train_part,
            params=params,
        )

        predictions = predict_model(
            model,
            val_part,
        )

        score = mape(
            val_part["num_sold"].to_numpy(),
            predictions,
        )

        candidate_scores.append(score)

        print(
            f"Fold {fold_number} ({year}): "
            f"MAPE = {score:.6f}%"
        )

    mean_score = float(np.mean(candidate_scores))

    tuning_results.append({
        "name": candidate_name,
        "iterations": params["iterations"],
        "depth": params["depth"],
        "learning_rate": params["learning_rate"],
        "l2_leaf_reg": params["l2_leaf_reg"],
        "mape_2014": candidate_scores[0],
        "mape_2015": candidate_scores[1],
        "mape_2016": candidate_scores[2],
        "mean_mape": mean_score,
    })

    print(
        f"Mean expanding-window MAPE: "
        f"{mean_score:.6f}%"
    )


Testing: v2_reference
{'iterations': 500, 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
Fold 1 (2014): MAPE = 7.334349%
Fold 2 (2015): MAPE = 13.599239%
Fold 3 (2016): MAPE = 6.565157%
Mean expanding-window MAPE: 9.166248%

Testing: shallower
{'iterations': 500, 'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
Fold 1 (2014): MAPE = 8.096333%
Fold 2 (2015): MAPE = 14.050403%
Fold 3 (2016): MAPE = 6.914035%
Mean expanding-window MAPE: 9.686923%

Testing: deeper
{'iterations': 500, 'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 3}


In [ ]:
tuning_results_df = (
    pd.DataFrame(tuning_results)
    .sort_values("mean_mape")
    .reset_index(drop=True)
)

print(
    tuning_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

In [ ]:
best_candidate = tuning_results_df.iloc[0]

print("\nBest configuration:")
print(best_candidate.to_string())

v2_mean_expanding_mape = (
    pd.read_csv(PROJECT_ROOT / "results" / "metrics.csv")
    .query(
        "model == 'catboost_log' "
        "and version == 'v2' "
        "and validation_scheme == 'expanding_window'"
    )["mape"]
    .mean()
)

improvement_vs_v2 = (
    v2_mean_expanding_mape
    - best_candidate["mean_mape"]
)

print("\nImprovement versus v2:")
print(
    f"{improvement_vs_v2:.6f} percentage points"
)

In [ ]:
best_params = {
    "iterations": int(best_candidate["iterations"]),
    "depth": int(best_candidate["depth"]),
    "learning_rate": float(best_candidate["learning_rate"]),
    "l2_leaf_reg": float(best_candidate["l2_leaf_reg"]),
}

print("Selected v3 parameters:")
print(best_params)

print("\nTraining v3 on long-horizon split...")

v3_long_model = fit_model(
    long_train,
    params=best_params,
)

v3_long_predictions = predict_model(
    v3_long_model,
    long_val,
)

v3_long_horizon_mape = mape(
    long_val["num_sold"].to_numpy(),
    v3_long_predictions,
)

print("Prediction min:", v3_long_predictions.min())
print("Prediction max:", v3_long_predictions.max())
print("Prediction mean:", v3_long_predictions.mean())
print(
    "Negative predictions:",
    int((v3_long_predictions < 0).sum()),
)
print(
    "All predictions finite:",
    bool(np.isfinite(v3_long_predictions).all()),
)

print(
    f"\nV3 long-horizon MAPE: "
    f"{v3_long_horizon_mape:.6f}%"
)

## 10. CatBoost v3: Tuned Model

The best hyperparameter configuration was selected using only the
expanding-window validation folds.

The selected configuration uses depth=8, while keeping 500 iterations,
learning_rate=0.05, and l2_leaf_reg=3.

The log1p target transformation and the existing feature set remain
unchanged from v2.

In [ ]:
V3_MODEL_NAME = "catboost_log_tuned"
V3_MODEL_VERSION = "v3"

v3_params = {
    "iterations": 500,
    "depth": 8,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3,
}

v3_fold_scores = []

for fold_number, (train_part, val_part) in enumerate(folds, start=1):
    year = int(val_part["date"].dt.year.iloc[0])
    fold_name = f"fold_{fold_number}_{year}"

    print(f"\nTraining {fold_name}")

    model = fit_model(
        train_part,
        params=v3_params,
    )

    predictions = predict_model(
        model,
        val_part,
    )

    score = mape(
        val_part["num_sold"].to_numpy(),
        predictions,
    )

    v3_fold_scores.append({
        "fold": fold_name,
        "mape": score,
    })

    print(f"MAPE: {score:.6f}%")

v3_fold_metrics = pd.DataFrame(v3_fold_scores)

v3_mean_expanding_mape = float(
    v3_fold_metrics["mape"].mean()
)

print("\nV3 expanding-window results:")
print(v3_fold_metrics.to_string(index=False))

print(
    f"\nV3 mean expanding-window MAPE: "
    f"{v3_mean_expanding_mape:.6f}%"
)

In [ ]:
import json

V3_MODEL_NAME = "catboost_log_tuned"
V3_MODEL_VERSION = "v3"

v3_experiment_params = {
    **v3_params,
    "features": FEATURE_COLS,
    "target_transform": TARGET_TRANSFORM,
    "prediction_postprocessing": "expm1_then_clip_min_0",
    "catboost_version": catboost.__version__,
}

v3_params_json = json.dumps(
    v3_experiment_params,
    sort_keys=True,
)

v3_evaluation_results = [
    (
        "expanding_window",
        row["fold"],
        train_part,
        val_part,
        float(row["mape"]),
    )
    for row, (train_part, val_part)
    in zip(v3_fold_scores, folds, strict=True)
]

v3_evaluation_results.append(
    (
        "long_horizon",
        "2014_2016",
        long_train,
        long_val,
        float(v3_long_horizon_mape),
    )
)

v3_records = []

for scheme, fold, train_part, val_part, score in v3_evaluation_results:
    v3_records.append({
        "model": V3_MODEL_NAME,
        "version": V3_MODEL_VERSION,
        "validation_scheme": scheme,
        "fold": fold,
        "train_period": (
            f"{train_part['date'].min().date()} to "
            f"{train_part['date'].max().date()}"
        ),
        "validation_period": (
            f"{val_part['date'].min().date()} to "
            f"{val_part['date'].max().date()}"
        ),
        "mape": round(score, 6),
        "params": v3_params_json,
    })

v3_metrics = pd.DataFrame(v3_records)

metrics_path = PROJECT_ROOT / "results" / "metrics.csv"
current_metrics = pd.read_csv(metrics_path)

assert list(v3_metrics.columns) == list(current_metrics.columns)
assert len(v3_metrics) == 4
assert np.isfinite(v3_metrics["mape"].to_numpy()).all()

print(
    v3_metrics
    .drop(columns="params")
    .to_string(index=False)
)

print("\nParameters:")
print(
    json.dumps(
        v3_experiment_params,
        indent=2,
        sort_keys=True,
    )
)

print("\nV3 metrics prepared. No files written.")

In [ ]:
current_metrics = pd.read_csv(metrics_path)
original_bytes = metrics_path.read_bytes()

own_rows = (
    current_metrics["model"].eq(V3_MODEL_NAME)
    & current_metrics["version"].eq(V3_MODEL_VERSION)
)

if own_rows.any():
    sort_columns = [
        "validation_scheme",
        "fold",
    ]

    existing = (
        current_metrics.loc[own_rows]
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )

    expected = (
        v3_metrics
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        existing,
        expected,
        check_dtype=False,
        check_exact=True,
    )

    print(
        "Identical v3 results already exist. "
        "No files changed."
    )

else:
    with metrics_path.open(
        "a",
        encoding="utf-8",
        newline="",
    ) as output:

        if (
            original_bytes
            and not original_bytes.endswith(
                (b"\n", b"\r")
            )
        ):
            output.write("\n")

        v3_metrics.to_csv(
            output,
            index=False,
            header=False,
            lineterminator="\n",
        )

    assert metrics_path.read_bytes().startswith(
        original_bytes
    )

    print(
        "Added 4 v3 rows. "
        "Existing rows preserved."
    )

updated_metrics = pd.read_csv(metrics_path)

print(
    "Total metric rows:",
    len(updated_metrics),
)

## 11. Final Model Comparison

In [ ]:
final_metrics = pd.read_csv(
    PROJECT_ROOT / "results" / "metrics.csv"
)

selected_models = final_metrics.loc[
    (
        final_metrics["model"].isin([
            "group_median",
            "seasonal_naive_last_year",
            "ridge_calendar_one_hot",
        ])
        & final_metrics["version"].eq("v1")
    )
    |
    (
        final_metrics["model"].eq("catboost_raw")
        & final_metrics["version"].eq("v1")
    )
    |
    (
        final_metrics["model"].eq("catboost_log")
        & final_metrics["version"].eq("v2")
    )
    |
    (
        final_metrics["model"].eq("catboost_log_tuned")
        & final_metrics["version"].eq("v3")
    )
].copy()

final_comparison = (
    selected_models
    .groupby(
        ["model", "version", "validation_scheme"]
    )["mape"]
    .mean()
    .unstack("validation_scheme")
)

model_order = [
    ("group_median", "v1"),
    ("ridge_calendar_one_hot", "v1"),
    ("seasonal_naive_last_year", "v1"),
    ("catboost_raw", "v1"),
    ("catboost_log", "v2"),
    ("catboost_log_tuned", "v3"),
]

final_comparison = final_comparison.reindex(
    model_order
)

print("Final model comparison:")
print(
    final_comparison
    .round(6)
    .to_string()
)

## 12. Conclusions

Three CatBoost model versions were evaluated using the same temporal
validation protocol and MAPE metric.

CatBoost v1 was trained directly on the original `num_sold` target and
achieved 18.24% mean expanding-window MAPE and 22.52% long-horizon MAPE.
It performed worse than the strongest baseline,
`seasonal_naive_last_year`, which achieved 13.27% and 17.02%,
respectively.

CatBoost v2 introduced a `log1p` transformation of the target while
keeping the feature set and model hyperparameters unchanged.
Predictions were transformed back with `expm1` before evaluation.
This substantially improved performance to 9.17% expanding-window MAPE
and 14.94% long-horizon MAPE, outperforming both CatBoost v1 and the
strongest baseline.

For CatBoost v3, a small hyperparameter search was performed using only
the expanding-window validation folds. The best configuration used
500 iterations, depth=8, learning_rate=0.05, and l2_leaf_reg=3.
The long-horizon holdout was evaluated only after the best configuration
had been selected.

CatBoost v3 achieved the best overall results:
9.04% mean expanding-window MAPE and 14.60% long-horizon MAPE.

Compared with v2, hyperparameter tuning provided a smaller additional
improvement of 0.12 percentage points on expanding-window validation and
0.34 percentage points on the long-horizon holdout. This indicates that
the main performance gain came from the log-target transformation,
while hyperparameter tuning provided a modest additional improvement.

Overall, CatBoost v3 is the best model evaluated in this experiment and
outperforms the strongest seasonal baseline on both validation schemes.
Further improvements, if required, should focus on additional
leakage-safe features rather than extensive hyperparameter tuning.